In [2]:
import pandas as pd

### 🧮 Las Funciones de Agregación Top (El Arsenal)

Estas son las funciones que vas a llamar constantemente después de agrupar tus datos.

| Función | ¿Qué calcula? | Caso de uso típico | Parámetro clave a recordar |
| --- | --- | --- | --- |
| **`sum()`** | La suma total. | Ingresos totales, stock total en bodega, suma de costos. | `numeric_only=True` (Evita intentar sumar columnas de texto, lo cual genera errores). |
| **`mean()`** | El promedio aritmético. | Ticket promedio de venta, precio promedio del mercado. | `numeric_only=True`. |
| **`median()`** | El valor central (Mediana). | **Mejor que el promedio** cuando hay valores atípicos (ej: un producto ridículamente caro que distorsiona el promedio). | Ninguno específico. |
| **`min()` / `max()**` | El valor mínimo / máximo. | Encontrar el precio más bajo, la fecha de la última compra (`max` en fechas). | Funciona tanto en números como en fechas o textos (orden alfabético). |
| **`count()`** | Cuenta filas **NO nulas**. | Saber cuántos precios válidos tienes (ignorando los `NaN`). | No cuenta los valores nulos (`NaN` o `None`). |
| **`size()`** | Cuenta **TODAS** las filas. | Saber el volumen total de transacciones, sin importar si faltan datos. | A diferencia de `count()`, es un atributo/método directo del grupo. |
| **`nunique()`** | Cuenta valores **únicos**. | Saber cuántos clientes distintos compraron, o cuántas marcas diferentes hay en una categoría. | `dropna=True` (Por defecto ignora los nulos al contar los únicos). |

---

### 💻 Cómo aplicarlas: Del nivel Básico al Senior

Existen diferentes formas de aplicar estas funciones dependiendo de la complejidad de tu análisis.

#### 1. Nivel Básico (Una columna, una métrica)

Se usa cuando necesitas una respuesta rápida para una sola métrica.


In [3]:
# Datos de ejemplo
df = pd.DataFrame({
    'categoria': ['Audio', 'Audio', 'Video', 'Video', 'Video'],
    'producto': ['Auricular A', 'Auricular B', 'Monitor X', 'Monitor Y', 'Webcam'],
    'precio': [50, 150, 200, 300, None], # Nota el None (nulo)
    'ventas': [10, 5, 2, 8, 15]
})

# ¿Cuál es el ingreso total (precio * ventas) por categoría?
df['ingreso'] = df['precio'] * df['ventas']
resumen_basico = df.groupby('categoria')['ingreso'].sum()

resumen_basico

categoria
Audio    1250.0
Video    2800.0
Name: ingreso, dtype: float64

#### 2. Nivel Intermedio (Varias métricas a la vez con `.agg()`)

Pasarle una lista a `.agg()` te permite ver diferentes perspectivas estadísticas de la misma columna. Ideal para análisis exploratorio.

In [5]:
# Queremos ver el precio mínimo, promedio y máximo por categoría
resumen_estadistico = df.groupby('categoria')['precio'].agg(['min', 'mean', 'max'])
resumen_estadistico

,min,mean,max
categoria,,,
Audio,50.0,100.0,150.0
Video,200.0,250.0,300.0


#### 3. Nivel Senior (Múltiples columnas, múltiples métricas, nombres limpios)

Esta es la **forma definitiva** de usar `groupby` en un pipeline de datos. Utilizas tuplas dentro de `.agg()` para aplicar diferentes funciones a diferentes columnas y, al mismo tiempo, bautizar la nueva columna resultante.

In [7]:
# Sintaxis: nombre_columna_nueva = ('columna_original', 'funcion_agregacion')
resumen_avanzado = df.groupby('categoria', as_index=False).agg(
    total_productos_distintos=('producto', 'nunique'),
    precio_mas_barato=('precio', 'min'),
    promedio_ventas=('ventas', 'mean'),
    registros_con_precio=('precio', 'count'), # Ignorará la Webcam porque su precio es nulo
    total_registros=('producto', 'size')      # Contará todo
)

print(resumen_avanzado)

  categoria  total_productos_distintos  precio_mas_barato  promedio_ventas  \
0     Audio                          2               50.0         7.500000   
1     Video                          3              200.0         8.333333   

   registros_con_precio  total_registros  
0                     2                2  
1                     2                3  


---

### 🚦 Diferencia Crítica: `count()` vs `size()`

Esta es una pregunta clásica en entrevistas técnicas de datos.

* **`size()`:** Es como el `COUNT(*)` de SQL. Te dice exactamente cuántas filas hay en ese grupo, punto.
* **`count()`:** Es como el `COUNT(columna)` de SQL. Cuenta cuántos datos reales hay, **descartando los nulos (`NaN`)**.

Si en tu grupo de "Video" tienes 3 productos pero uno no tiene precio registrado, `df.groupby('categoria')['precio'].size()` devolverá 3, pero `df.groupby('categoria')['precio'].count()` devolverá 2.
